# Machine Learning: Exercise session 04

In this exercise session we will focus on Ridge and Lasso regression. You will learn how to fit, predict, and cross-validate these models.

In the first problem, we will continue using the housing dataset where we added two additional variables `X1` and `X2`.
You can download the data from Moodle in this week's section.

In the second problem we will derive the closed-form solution for the Ridge regression (notice that Lasso regression has no closed-form solution).

## Problem 1

### 1. Load data and create a test set

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

Import the clean housing dataset (with `X1`, `X2`) using `pd.read_csv` and take a quick look at it, to verify that it is in the desired shape (display the dataframe, check its "infos").

In [ ]:
housing = pd.read_csv("housing_clean_with_X1X2.csv")

[autoreload of six failed: Traceback (most recent call last):
  File "/Users/ilorenci/Desktop/Autumn_2025/MachineLearning2025/.venv/lib/python3.12/site-packages/IPython/extensions/autoreload.py", line 325, in check
    superreload(m, reload, self.old_objects)
  File "/Users/ilorenci/Desktop/Autumn_2025/MachineLearning2025/.venv/lib/python3.12/site-packages/IPython/extensions/autoreload.py", line 621, in superreload
    update_generic(old_obj, new_obj)
  File "/Users/ilorenci/Desktop/Autumn_2025/MachineLearning2025/.venv/lib/python3.12/site-packages/IPython/extensions/autoreload.py", line 447, in update_generic
    update(a, b)
  File "/Users/ilorenci/Desktop/Autumn_2025/MachineLearning2025/.venv/lib/python3.12/site-packages/IPython/extensions/autoreload.py", line 380, in update_class
    old_obj = getattr(old, key)
              ^^^^^^^^^^^^^^^^^
  File "/Users/ilorenci/Desktop/Autumn_2025/MachineLearning2025/.venv/lib/python3.12/site-packages/six.py", line 98, in __get__
    setattr(o

In [ ]:
housing.head(10)

Separate the dataframe into the features `X` and the target variable `y`: (remember, we want to predict the median house value, given the other variables)

In [ ]:
X = housing.drop(["median_house_value"], axis = 1)
y = housing[["median_house_value"]]

Using the function `train_test_split`, split the dataset into training and test set. Set `test_size = 0.95` and `random_state = 12`. 

*Notice that we choose a very large fraction of test data because we want to see whether the Ridge and Lasso regression can handle the high-dimensional setting.*

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.95, random_state = 12)

### 2. Define models

Since ridge regression and lasso penalize the value of the coefficients $\beta_i$, it is important that all features are on a similar scale.
Is this the case here?

*Hint:* The output of `housing.describe()` gives a good overview over the mean, standard deviation, and relevant quantiles of each (numerical) feature.

In [ ]:
housing.describe()

Using the class `Ridge` from `sklearn.linear_model`, instantiate a ridge regression estimator with penalty parameter `alpha` set to `0.1` and `fit_intercept=True`.


In [ ]:
from sklearn.linear_model import Ridge
ridge_reg = Ridge(alpha = 0.1, fit_intercept = True)

Since the features are on very different scales, it is important that we scale them before performing further analysis.

To this end, create a `Pipeline` consisting of a `StandardScaler` instance and the `Ridge` instance from above.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

ridge_reg_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('ridge_reg', ridge_reg)
])

In [ ]:
ridge_reg_pipe

Using the class `Lasso` from `sklearn.linear_model`, instantiate a lasso regression object with penalty parameter `alpha` equal to `0.1` and `fit_intercept=True`.

Again, create a `Pipeline` from this object and a `StandardScaler` instance.

In [ ]:
from sklearn.linear_model import Lasso
lasso = Lasso(alpha = 0.1, fit_intercept = True)

lasso_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('lasso', lasso)
])

In [ ]:
lasso_pipe

What is the role of the option `fit_intercept`?

### 3. Fit Ridge regression

Fit the ridge regression (pipeline) on the training data previously created.

In [ ]:
ridge_reg_pipe.fit(X_train, y_train)

In [ ]:
ridge_reg = ridge_reg_pipe.named_steps['ridge_reg']

In [ ]:
ridge_reg.coef_

In [ ]:
ridge_reg_pipe.named_steps['ridge_reg'].coef_

Compute the root mean square error of the fitted model on both the training and test set. What do you observe?

In [ ]:
from sklearn.metrics import root_mean_squared_error
y_train_pred = ridge_reg_pipe.predict(X_train)
print("RMSE on training data:", root_mean_squared_error(y_train, y_train_pred))

y_test_pred = ridge_reg_pipe.predict(X_test)
print("RMSE on test data:", root_mean_squared_error(y_test, y_test_pred))

### 4. Plot Ridge regression coefficients

We now want to plot the coefficients of the ridge regression for different values of the penalty parameter.

Create a (geometrically spaced) grid, named `ridge_reg_grid`, of penalty values ranging from `1e-2` to `1e6`.

In [ ]:
ridge_reg_grid = 10 ** np.linspace(-2, 6, num = 30)
ridge_reg_grid = np.geomspace(1e-2, 1e6, num = 30)

Fill in the `??` below to create a list of the parameters found by the ridge regression pipeline for each `alpha` in `ridge_reg_grid`

In [ ]:
ridge_reg_pipe.set_params(ridge_reg__alpha = ridge_reg_grid[0])
ridge_reg_pipe.fit(X_train, y_train)

In [ ]:
coefs = []
for alpha in ridge_reg_grid:
    ridge_reg_pipe.set_params(ridge_reg__alpha = alpha)
    ridge_reg_pipe.fit(X_train, y_train)
    coefs.append(ridge_reg_pipe.named_steps['ridge_reg'].coef_)

In [ ]:
ridge_reg_pipe.named_steps['ridge_reg'].coef_

Plot the coefficients computed above.

In [ ]:
plt.plot(ridge_reg_grid, coefs)
plt.xlabel("$\\alpha$", fontsize = 14)
plt.ylabel("$\\beta_i$", fontsize = 14)
plt.xscale('log') # fits the geometric spacing of the grid
plt.legend(X_train.columns, loc='upper right', bbox_to_anchor = (1.5, 1))
plt.show()

In [ ]:
X_train.describe()

In [ ]:
# Illustration: what could happen if we do not use a scaler?

ridge_reg2 = Ridge(alpha = 0.1, fit_intercept = True)

X_train2 = X_train.copy()
# X_train2["median_income"] = X_train["median_income"] * 1e-0 # go down to 1e-6

coefs2 = []
for alpha in ridge_reg_grid:
    ridge_reg2.set_params(alpha = alpha)
    ridge_reg2.fit(X_train2, y_train)
    coefs2.append(ridge_reg2.coef_)

plt.plot(ridge_reg_grid, coefs2)
plt.xlabel("$\\alpha$", fontsize = 14)
plt.ylabel("$\\beta_i$", fontsize = 14)
plt.xscale('log') # fits the geometric spacing of the grid
plt.legend(X_train.columns, loc='upper right', bbox_to_anchor = (1.5, 1))
plt.show()



### 5. Fit Lasso regression

Fit the lasso regression (pipeline) on the training data previously created.

In [ ]:
lasso_pipe.fit(X_train, y_train)

Compute the root mean square error of the fitted model on both the training and test set. What do you observe?

In [ ]:
y_train_pred = lasso_pipe.predict(X_train)
print("RMSE on training data:", root_mean_squared_error(y_train, y_train_pred))

y_test_pred = lasso_pipe.predict(X_test)
print("RMSE on test data:", root_mean_squared_error(y_test, y_test_pred))

### 6. Plot Lasso coefficients

As above, we want to plot the coefficients of the lasso regression for different values of the penalty parameter.

Create a (geometrically spaced) grid, named `lasso_grid`, of penalty values ranging from `1` to `1e6`.

In [ ]:
lasso_grid = 10**np.linspace(0, 6, num=30)
lasso_grid = np.geomspace(1, 1e6, num=30)

* Fill in the `??` below.

In [ ]:
coefs = []
for alpha in lasso_grid:
    lasso_pipe.set_params(lasso__alpha=alpha)
    lasso_pipe.fit(X_train, y_train)
    coefs.append(lasso_pipe.named_steps['lasso'].coef_)

Plot the coefficients computed above.

In [ ]:
plt.plot(lasso_grid, coefs)
plt.xlabel("$\\alpha$", fontsize = 14)
plt.ylabel("$\\beta_i$", fontsize = 14)
plt.xscale('log')
plt.legend(X_train.columns, loc='upper right', bbox_to_anchor = (1.5, 1))
plt.show()

### 7. Prepare Cross Validation

At this point, we want to find the optimal penalty parameter for our models.
We will use 10-fold cross-validation.

Using the `KFold` class from `sklearn.model_selection` instatiate an object named `folds`.
Set the number of splits equal to 10, the random seed equal to 42, and make sure to shuffle the rows.

In [ ]:
from sklearn.model_selection import KFold
folds = KFold(n_splits = 10, random_state = 42, shuffle = True)

### 8. CV for Ridge

Create a dictionary with the key-value pair `"ridge_reg__alpha"`, `ridge_reg_grid`. What is the use of this object?

In [ ]:
ridge_reg_params = {"ridge_reg__alpha" : ridge_reg_grid}

Import the class `GridSearchCV` from `sklearn.model_selection` and use it to instantiate a **cross-validation object** for the ridge regression. Make sure to include the following parameters: `estimator`, `param_grid`, `scoring`, `cv`.

In [ ]:
from sklearn.model_selection import GridSearchCV
ridgeCV = GridSearchCV(
    estimator = ridge_reg_pipe,
    param_grid = ridge_reg_params,
    scoring = "neg_mean_squared_error",
    cv = folds
)

Run the cross-validation by calling the `fit` method of the **cross-validation object** that you created at the point before.

In [ ]:
ridgeCV.fit(X_train, y_train)

We want to plot the cross-validation error against different values of the penalty parameter. Fill in the `??`.

In [ ]:
pd.DataFrame(ridgeCV.cv_results_)

In [ ]:
# Choose best model that minimizes cv_err
mean_scores = -ridgeCV.cv_results_["mean_test_score"]
se_scores = ridgeCV.cv_results_["std_test_score"] / np.sqrt(ridgeCV.n_splits_)
alphas = ridgeCV.cv_results_["param_ridge_reg__alpha"].data

best_index = np.argmin(mean_scores)
min_alpha_ridge = alphas[best_index]
threshold_ridge = mean_scores[best_index] + se_scores[best_index]
one_se_rule_alpha_ridge = np.max(alphas[mean_scores <= threshold_ridge])

print("Minimum alpha:", min_alpha_ridge)
print("1-SD alpha:", one_se_rule_alpha_ridge)
print("Best score for ridge:", np.sqrt(np.min(mean_scores)))

In [ ]:
plt.errorbar(x=ridge_reg_grid, y=mean_scores, yerr = se_scores, fmt='o', capsize=3)

plt.axvline(min_alpha_ridge, ls='dotted', color="grey")#vertical line at the k yielding minimum CV MSE
plt.axvline(one_se_rule_alpha_ridge, ls='dotted', color="grey")
plt.axhline(threshold_ridge, ls='dotted', color="grey")

plt.title("Ridge regressor CV error")
plt.xlabel('log(alpha)')
plt.ylabel('Mean Squared Error')
plt.xscale('log')
plt.show()

### 9. CV for Lasso

Create a dictionary with the key-value pair `"lasso__alpha"`, `lasso_grid`. What is the use of this object?

In [ ]:
lasso_pipe.get_params()

In [ ]:
params_lasso = {"lasso__alpha" : lasso_grid}

Create a cross-validation object for the lasso regression. Make sure to specify the following parameters: `estimator`, `param_grid`, `scoring`, `cv`.

In [ ]:
lassoCV = GridSearchCV(
    estimator = lasso_pipe,
    param_grid = params_lasso,
    scoring = "neg_mean_squared_error",
    cv = folds
)

Run the cross-validation by calling the `fit` method of the **cross-validation object** that you created at the point before.

In [ ]:
lassoCV.fit(X_train, y_train)

We want to plot the cross-validation error against different values of the penalty parameter. Fill in the `??`.

In [ ]:
# Choose best model that minimizes the cv_err
mean_scores = -lassoCV.cv_results_["mean_test_score"]
se_scores = lassoCV.cv_results_["std_test_score"] / np.sqrt(lassoCV.n_splits_)
alphas = lassoCV.cv_results_["param_lasso__alpha"].data

best_index = np.argmin(mean_scores)
min_alpha_lasso = alphas[best_index]

threshold_lasso = mean_scores[best_index] + se_scores[best_index]
one_se_rule_alpha_lasso = np.max(alphas[mean_scores <= threshold_lasso])

print("Minimum alpha:", min_alpha_lasso)
print("1-SD alpha:", one_se_rule_alpha_lasso)
print("Best score for lasso:", np.sqrt(np.min(mean_scores)))

In [ ]:
plt.errorbar(x=lasso_grid, y=mean_scores, yerr = se_scores, fmt='o', capsize=3)

plt.axvline(min_alpha_lasso, ls='dotted', color="grey")
plt.axvline(one_se_rule_alpha_lasso, ls='dotted', color="grey")
plt.axhline(threshold_lasso, ls='dotted', color="grey")

plt.title("Lasso regressor CV error")
plt.xlabel('log(alpha)')
plt.ylabel('Mean Squared Error')
plt.xscale('log')
plt.show()

### 10. Compute performance of the two best models

Given the optimal tuning parameter of the ridge and lasso regression, refit both models on the entire training data set.
Furthermore, compute their root mean square error on the test set. How do they compare to the errors obtained before performing the cross-validation?


In [ ]:
# Refit best ridge on whole training data and evaluate on test
ridge_reg_pipe.set_params(ridge_reg__alpha = one_se_rule_alpha_ridge)
ridge_reg_pipe.fit(X_train, y_train)

y_test_pred = ridge_reg_pipe.predict(X_test)
root_mean_squared_error(y_test, y_test_pred)

In [ ]:
# Refit best lasso on whole training data and evaluate on test
lasso_pipe.set_params(lasso__alpha = one_se_rule_alpha_lasso)
lasso_pipe.fit(X_train, y_train)

y_test_pred = lasso_pipe.predict(X_test)
root_mean_squared_error(y_test, y_test_pred)

In [ ]:
# Show predictor that are kept/ignored by lasso
X.columns[lasso_pipe.named_steps['lasso'].coef_ != 0].to_list()
X.columns[lasso_pipe.named_steps['lasso'].coef_ == 0].to_list()

In [ ]:
ridge_reg_pipe.named_steps['ridge_reg'].coef_

In [ ]:
lasso_pipe.named_steps['lasso'].coef_

In [ ]:
# Linear regression
from sklearn.linear_model import LinearRegression
lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)

y_test_pred = lin_reg.predict(X_test)
root_mean_squared_error(y_test, y_test_pred)

## Problem 2

In this problem, you are asked to derive the closed-form solution for the ridge regression coefficients.

Let $\mathbf{X}\in\mathbb{R}^{n\times p}$ denote the matrix of predictors, and let $\mathbf{y}\in\mathbb{R}^n$ denote the target vector.

The optimal ridge regression coefficient vector $\beta^*$, with parameter $\lambda > 0$, is defined as

$$\beta^* := \arg\min_{\beta\in\mathbb{R}^p} \ (\mathbf{y} - \mathbf{X}\beta)^T(\mathbf{y} - \mathbf{X}\beta) + \lambda \beta ^T\beta.$$

Show that 


$$\beta^*  = (\mathbf{X}^T\mathbf{X} + \lambda \mathbf{I}_p)^{-1} \mathbf{X}^T \mathbf{y},$$

where $\mathbf{I}_p$ denote the identity matrix of size $p\times p$.


### Solution 
The loss function is defined, for $\lambda > 0$, as
\begin{align*}
L(\beta) 
= &\ (\mathbf{y} - \mathbf{X}\beta)^T(\mathbf{y} - \mathbf{X}\beta) + \lambda \beta ^T\beta\\
= &\ \mathbf{y}^T \mathbf{y} - 2\beta^T\mathbf{X}^T \mathbf y + \beta^T\mathbf{X}^T\mathbf{X}\beta + \lambda \beta^T \mathbf I_p \beta.
\end{align*}

To find the optimal ridge regression coefficient vector $\beta^*$, we need to differentiate the loss function with respect to $\beta$ and set it equal to zero.
The wikipedia article ["Matrix calculus"](https://en.wikipedia.org/wiki/Matrix_calculus#Scalar-by-vector_identities)
provides some useful identities for this computation.

This yields

\begin{align*}
0
&\stackrel{!}{=}
\frac{\partial L(\beta)}{\partial \beta} 
\\ &=
-2\mathbf{X}^T \mathbf y + 2 \mathbf{X}^T\mathbf{X}\beta + 2 \lambda \mathbf I_p \beta
\\ \Rightarrow \qquad
0
&\stackrel{!}{=}
\mathbf{X}^T \mathbf y -  (\mathbf{X}^T\mathbf{X} +  \lambda \mathbf I_p)\beta
\\ \Rightarrow \qquad
\beta^*
&=
(\mathbf{X}^T\mathbf{X} + \lambda \mathbf{I}_p)^{-1} \mathbf{X}^T \mathbf{y}
.
\end{align*}